In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# File handling
import joblib
import os

# Machine Learning
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    RandomizedSearchCV
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)

from scipy.stats import randint, uniform

# Reproducibility
RANDOM_STATE = 42

print("Day 07 environment initialized successfully.")
print("All required libraries imported.")


Day 07 environment initialized successfully.
All required libraries imported.


In [2]:

# Load original dataset
df = pd.read_csv("../dataset/student-mat.csv", sep=";")

# Load existing baseline artifacts
early_model = joblib.load(
    "../artifacts/beginning_term_model.pkl"
)

early_preprocessor = joblib.load(
    "../artifacts/beginning_term_preprocessor.pkl"
)

early_metrics = joblib.load(
    "../artifacts/beginning_term_model_metrics.pkl"
)

# Verify loaded data and artifacts
print("DATASET INFORMATION")
print("-" * 40)
print("Dataset shape:", df.shape)
print("Target column: G3")

print("\nBASELINE MODEL")
print("-" * 40)
print("Model:", early_metrics["Model"])
print("Original features:", early_metrics["Features"])
print("Transformed features:", early_metrics["Transformed Features"])

print("\nBASELINE PERFORMANCE")
print("-" * 40)
print(f"MAE: {early_metrics['MAE']:.4f}")
print(f"RMSE: {early_metrics['RMSE']:.4f}")
print(f"R²: {early_metrics['R2']:.4f}")
print(f"CV MAE: {early_metrics['CV_MAE']:.4f}")


DATASET INFORMATION
----------------------------------------
Dataset shape: (395, 33)
Target column: G3

BASELINE MODEL
----------------------------------------
Model: Beginning-of-Term Random Forest
Original features: 29
Transformed features: 55

BASELINE PERFORMANCE
----------------------------------------
MAE: 3.3423
RMSE: 4.1972
R²: 0.1409
CV MAE: 3.1809


In [3]:

# Separate features and target
excluded_columns = ["G1", "G2", "G3", "absences"]

X_early = df.drop(columns=excluded_columns)
y_early = df["G3"]

# Recreate the original train-test split
X_train_early, X_test_early, y_train_early, y_test_early = train_test_split(
    X_early,
    y_early,
    test_size=0.2,
    random_state=RANDOM_STATE
)

# Transform features using the saved preprocessor
X_train_processed = early_preprocessor.transform(X_train_early)
X_test_processed = early_preprocessor.transform(X_test_early)

# Verify dataset
print("TRAINING DATA")
print("-" * 35)
print("Features:", X_train_processed.shape)
print("Target:", y_train_early.shape)

print("\nTESTING DATA")
print("-" * 35)
print("Features:", X_test_processed.shape)
print("Target:", y_test_early.shape)

print("\nTotal transformed features:", X_train_processed.shape[1])


TRAINING DATA
-----------------------------------
Features: (316, 55)
Target: (316,)

TESTING DATA
-----------------------------------
Features: (79, 55)
Target: (79,)

Total transformed features: 55


In [4]:

# Predict using the existing baseline model
baseline_predictions = early_model.predict(X_test_processed)

# Calculate evaluation metrics
baseline_mae = mean_absolute_error(
    y_test_early, baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(y_test_early, baseline_predictions)
)

baseline_r2 = r2_score(
    y_test_early, baseline_predictions
)

print("BASELINE MODEL EVALUATION")
print("=" * 40)
print(f"MAE  : {baseline_mae:.4f}")
print(f"RMSE : {baseline_rmse:.4f}")
print(f"R²   : {baseline_r2:.4f}")


BASELINE MODEL EVALUATION
MAE  : 3.3423
RMSE : 4.1972
R²   : 0.1409


In [5]:

# Define hyperparameter search space
param_distributions = {
    "n_estimators": randint(100, 401),
    "max_depth": [3, 4, 5, 6, 8, 10, None],
    "min_samples_split": randint(2, 16),
    "min_samples_leaf": randint(1, 9),
    "max_features": uniform(0.5, 0.5)
}

# Initialize Random Forest
rf_model = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=1
)

# Configure Randomized Search
rf_random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_distributions,
    n_iter=40,
    scoring="neg_mean_absolute_error",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

# Perform hyperparameter optimization
rf_random_search.fit(
    X_train_processed,
    y_train_early
)

# Display results
print("\nBEST RANDOM FOREST PARAMETERS")
print("-" * 40)
print(rf_random_search.best_params_)

print("\nBEST CROSS-VALIDATION MAE")
print("-" * 40)
print(f"{-rf_random_search.best_score_:.4f}")


Fitting 5 folds for each of 40 candidates, totalling 200 fits

BEST RANDOM FOREST PARAMETERS
----------------------------------------
{'max_depth': 6, 'max_features': np.float64(0.522613644455269), 'min_samples_leaf': 8, 'min_samples_split': 6, 'n_estimators': 307}

BEST CROSS-VALIDATION MAE
----------------------------------------
3.1719


In [6]:

# Retrieve the best tuned model
best_rf_model = rf_random_search.best_estimator_

# Generate predictions on test data
tuned_predictions = best_rf_model.predict(X_test_processed)

# Calculate evaluation metrics
tuned_mae = mean_absolute_error(
    y_test_early, tuned_predictions
)

tuned_rmse = np.sqrt(
    mean_squared_error(y_test_early, tuned_predictions)
)

tuned_r2 = r2_score(
    y_test_early, tuned_predictions
)

# Compare against baseline
comparison = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Baseline": [
        baseline_mae,
        baseline_rmse,
        baseline_r2
    ],
    "Tuned Random Forest": [
        tuned_mae,
        tuned_rmse,
        tuned_r2
    ]
})

print("BASELINE VS TUNED RANDOM FOREST")
print("=" * 60)
print(comparison.to_string(index=False))

print("\nPerformance difference:")
print(f"MAE improvement: {baseline_mae - tuned_mae:.4f}")
print(f"RMSE improvement: {baseline_rmse - tuned_rmse:.4f}")
print(f"R2 improvement: {tuned_r2 - baseline_r2:.4f}")



BASELINE VS TUNED RANDOM FOREST
Metric  Baseline  Tuned Random Forest
   MAE  3.342315             3.252383
  RMSE  4.197195             4.079832
    R2  0.140873             0.188247

Performance difference:
MAE improvement: 0.0899
RMSE improvement: 0.1174
R2 improvement: 0.0474


In [7]:

# Define alternative regression models
alternative_models = {
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        min_samples_leaf=5,
        random_state=RANDOM_STATE
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=300,
        max_depth=6,
        min_samples_split=6,
        min_samples_leaf=8,
        max_features=0.7,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "Hist Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=RANDOM_STATE
    )
}

# Store model evaluation results
alternative_results = []
trained_models = {}

# Train and evaluate each model
for name, model in alternative_models.items():

    model.fit(X_train_processed, y_train_early)

    predictions = model.predict(X_test_processed)

    mae = mean_absolute_error(y_test_early, predictions)
    rmse = np.sqrt(mean_squared_error(y_test_early, predictions))
    r2 = r2_score(y_test_early, predictions)

    alternative_results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

    trained_models[name] = model

    print(f"\n{name}")
    print("-" * 35)
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")

print("\nAll alternative models trained successfully.")



Gradient Boosting
-----------------------------------
MAE  : 3.4606
RMSE : 4.2991
R2   : 0.0987

Extra Trees
-----------------------------------
MAE  : 3.2990
RMSE : 4.1503
R2   : 0.1600

Hist Gradient Boosting
-----------------------------------
MAE  : 3.6954
RMSE : 4.4426
R2   : 0.0375

All alternative models trained successfully.


In [8]:

# Collect all models for cross-validation
models_for_cv = {
    "Original Random Forest": early_model,
    "Tuned Random Forest": best_rf_model,
    **trained_models
}

cv_results = []

for name, model in models_for_cv.items():

    scores = cross_val_score(
        model,
        X_train_processed,
        y_train_early,
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    cv_mae_scores = -scores

    cv_results.append({
        "Model": name,
        "CV MAE": cv_mae_scores.mean(),
        "CV Std": cv_mae_scores.std()
    })

    print(f"{name}: CV MAE = {cv_mae_scores.mean():.4f} "
          f"(Std = {cv_mae_scores.std():.4f})")

# Create comparison table
cv_comparison = pd.DataFrame(cv_results)

print("\nCROSS-VALIDATION COMPARISON")
print("=" * 65)

print(
    cv_comparison
    .sort_values("CV MAE")
    .to_string(index=False)
)


Original Random Forest: CV MAE = 3.1809 (Std = 0.2947)
Tuned Random Forest: CV MAE = 3.1719 (Std = 0.2508)
Gradient Boosting: CV MAE = 3.5285 (Std = 0.3777)
Extra Trees: CV MAE = 3.2697 (Std = 0.2639)
Hist Gradient Boosting: CV MAE = 3.4365 (Std = 0.3534)

CROSS-VALIDATION COMPARISON
                 Model   CV MAE   CV Std
   Tuned Random Forest 3.171948 0.250759
Original Random Forest 3.180872 0.294669
           Extra Trees 3.269741 0.263886
Hist Gradient Boosting 3.436515 0.353386
     Gradient Boosting 3.528456 0.377685


In [9]:

# Prepare test performance results
test_results = [
    {
        "Model": "Original Random Forest",
        "Test MAE": baseline_mae,
        "Test RMSE": baseline_rmse,
        "Test R2": baseline_r2
    },
    {
        "Model": "Tuned Random Forest",
        "Test MAE": tuned_mae,
        "Test RMSE": tuned_rmse,
        "Test R2": tuned_r2
    }
]

# Add alternative model results
for result in alternative_results:
    test_results.append({
        "Model": result["Model"],
        "Test MAE": result["MAE"],
        "Test RMSE": result["RMSE"],
        "Test R2": result["R2"]
    })

test_comparison = pd.DataFrame(test_results)

# Merge test and cross-validation results
final_comparison = test_comparison.merge(
    cv_comparison,
    on="Model"
)

# Sort by test MAE for presentation
final_comparison = final_comparison.sort_values(
    by="Test MAE"
).reset_index(drop=True)

print("COMPLETE MODEL PERFORMANCE COMPARISON")
print("=" * 85)

print(final_comparison.round(4).to_string(index=False))

COMPLETE MODEL PERFORMANCE COMPARISON
                 Model  Test MAE  Test RMSE  Test R2  CV MAE  CV Std
   Tuned Random Forest    3.2524     4.0798   0.1882  3.1719  0.2508
           Extra Trees    3.2990     4.1503   0.1600  3.2697  0.2639
Original Random Forest    3.3423     4.1972   0.1409  3.1809  0.2947
     Gradient Boosting    3.4606     4.2991   0.0987  3.5285  0.3777
Hist Gradient Boosting    3.6954     4.4426   0.0375  3.4365  0.3534


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Identify numerical and categorical columns
numeric_features = X_early.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_early.select_dtypes(
    exclude=["number"]
).columns.tolist()

# Build preprocessing inside the pipeline
cv_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ]
)

# Build the optimized Random Forest pipeline
leakage_safe_pipeline = Pipeline(
    steps=[
        ("preprocessor", cv_preprocessor),
        ("model", RandomForestRegressor(
            **rf_random_search.best_params_,
            random_state=RANDOM_STATE,
            n_jobs=1
        ))
    ]
)

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total original features:", len(X_early.columns))
print("Pipeline created successfully.")

Numerical features: 12
Categorical features: 17
Total original features: 29
Pipeline created successfully.


In [11]:
leakage_safe_scores = cross_val_score(
    leakage_safe_pipeline,
    X_train_early,
    y_train_early,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

leakage_safe_mae = -leakage_safe_scores

print("LEAKAGE-AWARE CROSS-VALIDATION RESULTS")
print("=" * 50)
print("Fold MAE scores:", np.round(leakage_safe_mae, 4))
print(f"Mean CV MAE: {leakage_safe_mae.mean():.4f}")
print(f"CV Standard Deviation: {leakage_safe_mae.std():.4f}")

LEAKAGE-AWARE CROSS-VALIDATION RESULTS
Fold MAE scores: [3.45   3.0164 3.0942 3.4705 2.829 ]
Mean CV MAE: 3.1720
CV Standard Deviation: 0.2507


In [12]:
# Fit the complete pipeline on all available training data
leakage_safe_pipeline.fit(X_train_early, y_train_early)

# Save optimized pipeline
optimized_model_path = "../artifacts/day07_optimized_pipeline.pkl"

joblib.dump(leakage_safe_pipeline, optimized_model_path)

# Save validation metrics
day07_metrics = {
    "Model": "Optimized Random Forest Pipeline",
    "CV_MAE": leakage_safe_mae.mean(),
    "CV_Std": leakage_safe_mae.std(),
    "Test_MAE": tuned_mae,
    "Test_RMSE": tuned_rmse,
    "Test_R2": tuned_r2
}

day07_metrics_df = pd.DataFrame([day07_metrics])

day07_metrics_df.to_csv(
    "../artifacts/day07_optimization_metrics.csv",
    index=False
)

print("Optimized pipeline saved successfully.")
print("Metrics saved successfully.")
print("\nFinal model metrics:")
display(day07_metrics_df)

Optimized pipeline saved successfully.
Metrics saved successfully.

Final model metrics:


,Model,CV_MAE,CV_Std,Test_MAE,Test_RMSE,Test_R2
0,Optimized Random Forest Pipeline,3.172016,0.250717,3.252383,4.079832,0.188247


In [13]:
# Reload the saved pipeline
loaded_optimized_pipeline = joblib.load(
    "../artifacts/day07_optimized_pipeline.pkl"
)

# Select a few raw test records
sample_students = X_test_early.head(5)

# Generate predictions using raw features
sample_predictions = loaded_optimized_pipeline.predict(
    sample_students
)

# Display predictions
verification_df = pd.DataFrame({
    "Actual G3": y_test_early.head(5).values,
    "Predicted G3": np.round(sample_predictions, 2)
})

verification_df["Absolute Error"] = np.abs(
    verification_df["Actual G3"] -
    verification_df["Predicted G3"]
)

display(verification_df)

print("\nPipeline reload verification successful.")
print("Input shape:", sample_students.shape)
print("Prediction count:", len(sample_predictions))

,Actual G3,Predicted G3,Absolute Error
0,10,5.88,4.12
1,12,9.01,2.99
2,5,7.93,2.93
3,10,10.60,0.60
4,9,8.30,0.70



Pipeline reload verification successful.
Input shape: (5, 29)
Prediction count: 5
